In [1]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install --only-binary=:all: "jpype1>=1.5"
!{sys.executable} -m pip -q install neo4j pandas python-dotenv 
!{sys.executable} -m pip install -q --only-binary=jpype1 pslpython
!{sys.executable} -m pip install pyvis

In [2]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import neo4j
import pandas as pd
import os

load_dotenv()

True

In [162]:
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()


In [4]:
def run_cypher(query: str, params: dict | None = None) -> pd.DataFrame:
    params = params or {}
    with driver.session() as session:
        res = session.run(query, params)
        rows = [r.data() for r in res]
    return pd.DataFrame(rows)

In [5]:
labels = run_cypher("CALL db.labels() YIELD label RETURN label ORDER BY label;")
rels = run_cypher("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType;")
labels, rels

(      label
 0      Case
 1     Claim
 2    Entity
 3     Event
 4  Evidence
 5  Location
 6    Person,
            relationshipType
 0            ACCOMPANIED_BY
 1      ACCOMPANIED_ON_NIGHT
 2                ACCUSED_IN
 3               ANALYZED_AT
 4         ATTENDED_PARTY_AT
 5           EVIDENCE_SEIZED
 6        FAMILY_INVOLVED_IN
 7                 FAMILY_OF
 8       FAMILY_RELATIONSHIP
 9       FILED_CIVIL_SUIT_IN
 10                 FOUND_AT
 11             INVESTIGATED
 12          INVESTIGATED_BY
 13          INVESTIGATED_IN
 14             LAST_SEEN_AT
 15           LAST_SEEN_NEAR
 16                 LIVED_AT
 17  LIVED_IN_OR_WORKED_NEAR
 18        MISSING_PERSON_IN
 19       PERSON_OF_INTEREST
 20      POTENTIAL_VICTIM_IN
 21             PRESENT_NEAR
 22               RESCUED_AT
 23               RESCUED_IN
 24             RESIDENCE_OF
 25              SEARCHED_AT
 26              SEIZED_FROM
 27                SEIZED_IN
 28             STATIONED_AT
 29                STAYED

In [6]:
samplepersons = run_cypher("MATCH (n:Person) RETURN keys(n) AS person_props, n LIMIT 5")
samplecases = run_cypher("MATCH (n:Case) RETURN keys(n) AS case_props, n LIMIT 5")
samplelocs = run_cypher("MATCH (n:Location) RETURN keys(n) AS loc_props, n LIMIT 5")
sampleevents = run_cypher("MATCH (n:Event) RETURN keys(n) AS event_props, n LIMIT 5")
sampleentities = run_cypher("MATCH (n:Entity) RETURN keys(n) AS entity_props, n LIMIT 5")
sampleevidence = run_cypher("MATCH (n:Evidence) RETURN keys(n) AS evidence_props, n LIMIT 5")

samplepersons, samplecases, samplelocs, sampleevents, sampleentities, sampleevidence

(                    person_props  \
 0       [type, status, id, name]   
 1       [type, status, id, name]   
 2  [status, type, dob, id, name]   
 3       [type, status, id, name]   
 4  [dob, type, status, id, name]   
 
                                                    n  
 0  {'name': 'Tokyo Man 1', 'id': 'TOKYO_MAN_1', '...  
 1  {'name': 'Tokyo Man 2', 'id': 'TOKYO_MAN_2', '...  
 2  {'dob': '1959-??-??', 'name': 'Kenji Iwamura',...  
 3  {'name': 'Unknown Female', 'id': 'UNKNOWN_FEMA...  
 4  {'dob': '2003-05-12', 'name': 'Madeleine McCan...  ,
                        case_props  \
 0  [name, status, dateOpened, id]   
 1  [status, name, dateOpened, id]   
 2  [status, dateOpened, id, name]   
 3  [name, dateOpened, id, status]   
 4  [name, dateOpened, id, status]   
 
                                                    n  
 0  {'dateOpened': '1989-07-24', 'name': 'Mountain...  
 1  {'dateOpened': '2007-05-03', 'name': 'Disappea...  
 2  {'dateOpened': '1995-06-27', 'name': 

### Visualize Database

In [7]:
from pyvis.network import Network

query = """
MATCH p=()-[r]->()
RETURN p
LIMIT 200
"""

net = Network(
    height="750px",
    width="100%",
    directed=True,
    notebook=False,
    cdn_resources="in_line",
)

seen_nodes = set()
seen_edges = set()

with driver.session() as session:
    result = session.run(query)

    count_paths = 0
    for record in result:
        path = record["p"]
        count_paths += 1

        for node in path.nodes:
            node_id = node.element_id
            node_label = next(iter(node.labels), "Node")
            display = node.get("name") or node.get("id") or node_label

            if node_id not in seen_nodes:
                net.add_node(
                    node_id,
                    label=display,
                    title=str(dict(node)),
                    group=node_label,
                )
                seen_nodes.add(node_id)

        for rel in path.relationships:
            edge_key = (rel.element_id,)
            if edge_key not in seen_edges:
                net.add_edge(
                    rel.start_node.element_id,
                    rel.end_node.element_id,
                    label=rel.type,
                    title=rel.type,
                )
                seen_edges.add(edge_key)

print("paths:", count_paths)
print("nodes:", len(seen_nodes))
print("edges:", len(seen_edges))

net.write_html("graph.html")

paths: 74
nodes: 58
edges: 74


### Victim Related Individuals In Knowledge Graph

In [163]:
q = """
MATCH (p:Person)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
OPTIONAL MATCH (p)--(nbr)
WITH
  c, p, r,
  count(DISTINCT nbr) AS person_degree
RETURN
  c.id AS case_id,
  c.name AS case_name,
  p.id AS person_id,
  p.name AS person_name,
  p.dob AS person_dob,
  p.type AS person_type,
  type(r) AS relationship_type,
  person_degree
LIMIT 500
"""
df_victim_individuals = run_cypher(q)
df_victim_individuals.head()

,case_id,case_name,person_id,person_name,person_dob,person_type,relationship_type,person_degree
0,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,UNKNOWN_FEMALE,Unknown Female,NaN,Mountaineer,POTENTIAL_VICTIM_IN,1
1,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,KI,Kenji Iwamura,1959-??-??,Mountaineer,VICTIM_IN,2
2,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,TOKYO_MAN_2,Tokyo Man 2,NaN,Mountaineer,RESCUED_IN,2
3,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,TOKYO_MAN_1,Tokyo Man 1,NaN,Mountaineer,RESCUED_IN,2
4,CASE_MM,Disappearance of Madeleine McCann,MM,Madeleine McCann,2003-05-12,Victim,MISSING_PERSON_IN,4


### Suspect Related Individuals In Knowledge Graph

In [149]:
q = """
MATCH (p:Person)-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c:Case)
OPTIONAL MATCH (p)--(nbr)
WITH
  c, p, r,
  count(DISTINCT nbr) AS person_degree
RETURN
  c.id AS case_id,
  c.name AS case_name,
  p.id AS person_id,
  p.name AS person_name,
  p.dob AS person_dob,
  p.type AS person_type,
  type(r) AS relationship_type,
  person_degree
LIMIT 500
"""
df_suspects_in_cases = run_cypher(q)
df_suspects_in_cases.head()

,case_id,case_name,person_id,person_name,person_dob,person_type,relationship_type,person_degree
0,CASE_MM,Disappearance of Madeleine McCann,CB,Christian Brueckner,1976-12-07,Suspect,SUSPECT_IN,2
1,CASE_KS,Murder of Kristin Smart,RF,Ruben Flores,1941-01-01,Suspect/Accessory,ACCUSED_IN,3
2,CASE_KS,Murder of Kristin Smart,PF,Paul Flores,1977-04-11,Suspect/Murderer,SUSPECT_IN,8
3,CASE_MB,Killing of Molly Bish,FS,Francis 'Frank' Sumner Sr.,,Person of Interest,PERSON_OF_INTEREST,3


### Hotspot Query

In [10]:
q = """
MATCH (c:Case)<-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]-(p:Person)
MATCH (p)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
WHERE loc.name IS NOT NULL OR loc.city IS NOT NULL
WITH c, coalesce(loc.name, loc.city) AS place
WHERE place IS NOT NULL
RETURN place AS location, count(DISTINCT c) AS case_count
ORDER BY case_count DESC
LIMIT 50
"""
df_hotspots = run_cypher(q)
df_hotspots.head()

,location,case_count
0,SOS Sign Discovery Site,1
1,Mount Kurodake to Mount Asahi Path,1
2,APT_5A,1
3,Santa Lucia Hall (PF's Dorm),1
4,Comins Pond,1


### Find Victim Related Individuals

In [150]:
q = """
MATCH (c:Case)
OPTIONAL MATCH (p:Person)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
WITH
  c,
  count(DISTINCT p) AS persons_linked,
  collect(DISTINCT type(r)) AS relationship_types
RETURN
  c.id AS case_id,
  c.name AS case_name,
  c.status AS case_status,
  persons_linked,
  relationship_types
LIMIT 100
"""
df_case_summary = run_cypher(q)
df_case_summary.head()

Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (c:Case)\nOPTIONAL MATCH (p:Person)-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)\nWITH\n  c,\n  count(DISTINCT p) AS persons_linked,\n  collect(DISTINCT type(r)) AS relationship_types\nRETURN\n  c.id AS case_id,\n  c.name AS case_name,\n  c.status AS case_status,\n  persons_linked,\n  relationship_types\nLIMIT 100\n'


,case_id,case_name,case_status,persons_linked,relationship_types
0,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,Open/Investigated,4,"[POTENTIAL_VICTIM_IN, VICTIM_IN, RESCUED_IN]"
1,CASE_MM,Disappearance of Madeleine McCann,Open (Missing),1,[MISSING_PERSON_IN]
2,C1,Disappearance of Jodi Sue Huisentruit,Open missing person / homicide investigation,0,[]
3,C2,Murder of Deidre Harm,Closed; offender identified (Revak),0,[]
4,C3,Murder of Rene Williams,Open homicide (suspect deceased before trial),0,[]


### Query Individual Related Cases

In [64]:
def query_victim_individual(id: str) -> pd.DataFrame:
    q = """
        MATCH (p:Person {id: $personId})-[r:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        RETURN
        p.id AS personId,
        c.id AS caseId,
        collect(DISTINCT type(r)) AS relationship_types
        ORDER BY caseId
        LIMIT 50
    """
    return run_cypher(q, {"personId": id})

def query_suspect_individual(id: str) -> pd.DataFrame:
    q = """
        MATCH (p:Person {id: $personId})-[r:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c:Case)
        RETURN
        p.id AS personId,
        c.id AS caseId,
        collect(DISTINCT type(r)) AS relationship_types
        ORDER BY caseId
        LIMIT 50
    """
    return run_cypher(q, {"personId": id})


In [164]:
query_victim_individual("UNKNOWN_FEMALE")

,personId,caseId,relationship_types
0,UNKNOWN_FEMALE,CASE_MOUNTAIN,[POTENTIAL_VICTIM_IN]


In [73]:
query_suspect_individual("UNKNOWN_FEMALE")

""


In [68]:
query_victim_individual("CB")

""


In [165]:
query_suspect_individual("CB")

,personId,caseId,relationship_types
0,CB,CASE_MM,[SUSPECT_IN]


In [70]:
query_victim_individual("TJGE")

,personId,caseId,relationship_types
0,TJGE,CASE_TJGE,[VICTIM_IN]


In [ ]:
query_suspect_individual("TJGE")

### Query Individual Related Locations

In [153]:
def query_victim_individual_location(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[rc:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        OPTIONAL MATCH (p)-[rl:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        RETURN
            p.id AS personId,
            c.id AS caseId,
            type(rl) AS linkType,
            loc.id AS locationId,
            loc.name AS locationName
        ORDER BY caseId
        LIMIT 50
    """

    return run_cypher(q, {"personId": id})

def query_suspect_individual_location(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[rc:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c:Case)
        OPTIONAL MATCH (p)-[rl:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        RETURN
            p.id AS personId,
            c.id AS caseId,
            type(rc) AS personCaseRel,
            type(rl) AS linkType,
            loc.id AS locationId,
        loc.name AS locationName
        ORDER BY caseId
        LIMIT 50
    """

    return run_cypher(q, {"personId": id})

In [167]:
query_victim_individual_location("UNKNOWN_FEMALE")

,personId,caseId,linkType,locationId,locationName
0,UNKNOWN_FEMALE,CASE_MOUNTAIN,None,None,None


In [81]:
query_suspect_individual_location("KI")

""


In [155]:
query_victim_individual_location("MM")

,personId,caseId,linkType,locationId,locationName
0,MM,CASE_MM,LAST_SEEN_AT,APT_5A,APT_5A


In [83]:
query_suspect_individual_location("MM")

""


In [84]:
query_victim_individual_location("CB")

""


In [85]:
query_suspect_individual_location("CB")

,personId,caseId,personCaseRel,linkType,locationId,locationName
0,CB,CASE_MM,SUSPECT_IN,PRESENT_NEAR,PRAIA_DA_LUZ,PRAIA_DA_LUZ


### Score Individual's Locations

In [156]:
def score_victim_individual(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[r1:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        OPTIONAL MATCH (p)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)

        WITH p, c, loc,
            type(r1) AS personCaseRel,
            type(r2) AS caseLocRel,

            CASE type(r1)
            WHEN 'VICTIM_IN' THEN 1.0
            WHEN 'MISSING_PERSON_IN' THEN 0.9
            WHEN 'RESCUED_IN' THEN 0.85
            WHEN 'POTENTIAL_VICTIM_IN' THEN 0.6
            ELSE 0.3
            END AS s1,

            CASE type(r2)
            WHEN 'FOUND_AT' THEN 1.0
            WHEN 'LAST_SEEN_AT' THEN 0.95
            WHEN 'RESCUED_AT' THEN 0.9
            WHEN 'LAST_SEEN_NEAR' THEN 0.75
            WHEN 'PRESENT_NEAR' THEN 0.7
            WHEN 'STAYED_AT' THEN 0.65
            WHEN 'LIVED_AT' THEN 0.5
            WHEN 'LIVED_IN_OR_WORKED_NEAR' THEN 0.4
            ELSE 0.3
            END AS s2

        RETURN
        p.id AS personId,
        c.id AS caseId,
        loc.id AS locationId,
        loc.name AS locationName,
        personCaseRel,
        caseLocRel,
        s1 * s2 AS pathScore

        ORDER BY pathScore DESC, locationName
    """

    return run_cypher(q, {"personId": id})

def score_suspect_individual(id: str):
    q = """
        MATCH (p:Person {id: $personId})
        MATCH (p)-[r1:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c:Case)
        OPTIONAL MATCH (p)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)

        WITH p, c, loc,
            type(r1) AS personCaseRel,
            type(r2) AS caseLocRel,

            CASE type(r1)
            WHEN 'SUSPECT_IN' THEN 1.0
            WHEN 'ACCUSED_IN' THEN 0.9
            WHEN 'PERSON_OF_INTEREST' THEN 0.6
            ELSE 0.3
            END AS s1,

            CASE type(r2)
            WHEN 'FOUND_AT' THEN 1.0
            WHEN 'LAST_SEEN_AT' THEN 0.95
            WHEN 'RESCUED_AT' THEN 0.9
            WHEN 'LAST_SEEN_NEAR' THEN 0.75
            WHEN 'PRESENT_NEAR' THEN 0.7
            WHEN 'STAYED_AT' THEN 0.65
            WHEN 'LIVED_AT' THEN 0.5
            WHEN 'LIVED_IN_OR_WORKED_NEAR' THEN 0.4
            ELSE 0.3
            END AS s2

        RETURN
        p.id AS personId,
        c.id AS caseId,
        loc.id AS locationId,
        loc.name AS locationName,
        personCaseRel,
        caseLocRel,
        s1 * s2 AS pathScore

        ORDER BY pathScore DESC, locationName
    """

    return run_cypher(q, {"personId": id})

In [168]:
score_victim_individual("KI")

,personId,caseId,locationId,locationName,personCaseRel,caseLocRel,pathScore
0,KI,CASE_MOUNTAIN,SOS_SIGN_HOLE,SOS Sign Discovery Site,VICTIM_IN,FOUND_AT,1.0


In [90]:
score_suspect_individual("KI")

""


In [169]:
score_victim_individual("TOKYO_MAN_1")

,personId,caseId,locationId,locationName,personCaseRel,caseLocRel,pathScore
0,TOKYO_MAN_1,CASE_MOUNTAIN,DAISETSUZAN_PATH,Mount Kurodake to Mount Asahi Path,RESCUED_IN,RESCUED_AT,0.765


In [92]:
score_suspect_individual("TOKYO_MAN_1")

""


In [93]:
score_victim_individual("CB")

""


In [94]:
score_suspect_individual("CB")

,personId,caseId,locationId,locationName,personCaseRel,caseLocRel,pathScore
0,CB,CASE_MM,PRAIA_DA_LUZ,PRAIA_DA_LUZ,SUSPECT_IN,PRESENT_NEAR,0.7


### Find Related Individuals

In [96]:
def find_related_victim_individuals(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        MATCH (target)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c)
        MATCH (other)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
        other.id AS similarPersonId,
        count(DISTINCT c) AS sharedCases,
        count(DISTINCT loc) AS sharedLocations,
        collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedCases DESC, sharedLocations DESC
        LIMIT 25
    """
    return run_cypher(q, {"personId": id})

def find_related_suspect_individuals(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c:Case)
        MATCH (target)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(c)
        MATCH (other)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
        other.id AS similarPersonId,
        count(DISTINCT c) AS sharedCases,
        count(DISTINCT loc) AS sharedLocations,
        collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedCases DESC, sharedLocations DESC
        LIMIT 25
    """
    return run_cypher(q, {"personId": id})

In [170]:
find_related_victim_individuals("TOKYO_MAN_1")

,similarPersonId,sharedCases,sharedLocations,overlappingLocations
0,TOKYO_MAN_2,1,1,[Mount Kurodake to Mount Asahi Path]


In [98]:
find_related_suspect_individuals("TOKYO_MAN_1")

""


In [99]:
find_related_victim_individuals("CB")

""


In [100]:
find_related_suspect_individuals("CB")

""


### Find Related Individuals To Individual Location

In [101]:
def related_individuals_to_victim_locations(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
            other.id AS similarPersonId,
            count(DISTINCT loc) AS sharedLocations,
            collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedLocations DESC
        LIMIT 25
    """

    return run_cypher(q, {"personId": id})

def related_individuals_to_suspect_locations(id: str) -> pd.DataFrame:
    q = """
        MATCH (target:Person {id: $personId})-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(:Case)
        MATCH (target)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location)
        MATCH (other:Person)-[:ACCUSED_IN|PERSON_OF_INTEREST|SUSPECT_IN]->(:Case)
        MATCH (other)-[:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|LIVED_AT|LIVED_IN_OR_WORKED_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc)
        WHERE other <> target
        RETURN
            other.id AS similarPersonId,
            count(DISTINCT loc) AS sharedLocations,
            collect(DISTINCT coalesce(loc.name, loc.city, loc.id))[0..10] AS overlappingLocations
        ORDER BY sharedLocations DESC
        LIMIT 25
    """

    return run_cypher(q, {"personId": id})

In [172]:
related_individuals_to_victim_locations("TOKYO_MAN_1")

,similarPersonId,sharedLocations,overlappingLocations
0,TOKYO_MAN_2,1,[Mount Kurodake to Mount Asahi Path]


In [ ]:
related_individuals_to_suspect_locations("MM")

""


In [104]:
related_individuals_to_victim_locations("CB")

""


In [105]:
related_individuals_to_suspect_locations("CB")

""


### Find Relations Of Individual At Location

In [107]:
def query_victim_individual_at_location(personId : str, locationId) -> pd.DataFrame:
    q = """
        MATCH (person:Person {id: $personId})-[r1:MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN|VICTIM_IN]->(c:Case)
        MATCH (person)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location {id: $locationId})

        RETURN
            person.id AS personId,
            person.name AS personName,
            person.dob AS personDOB,
            person.type AS personType,

            c.id AS caseId,
            c.name AS caseName,
            c.status AS caseStatus,

            loc.id AS locationId,
            loc.name AS locationName,
            loc.city AS locationCity,

            type(r1) AS personCaseRel,
            type(r2) AS personLocationRel
        LIMIT 10
    """

    return run_cypher(q, {"personId": personId, "locationId": locationId})

def query_suspect_individual_at_location(personId: str, locationId: str) -> pd.DataFrame:
    q = """
        MATCH (person:Person {id: $personId})-[r1:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c:Case)
        MATCH (person)-[r2:FOUND_AT|LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|RESCUED_AT|STAYED_AT]->(loc:Location {id: $locationId})

        RETURN
            person.id AS personId,
            person.name AS personName,
            person.dob AS personDOB,
            person.type AS personType,

            c.id AS caseId,
            c.name AS caseName,
            c.status AS caseStatus,

            loc.id AS locationId,
            loc.name AS locationName,
            loc.city AS locationCity,

            type(r1) AS personCaseRel,
            type(r2) AS personLocationRel
        LIMIT 10
    """

    return run_cypher(q, {
        "personId": personId,
        "locationId": locationId
    })

In [173]:
query_victim_individual_at_location("KI", "SOS_SIGN_HOLE")

,personId,personName,personDOB,personType,caseId,caseName,caseStatus,locationId,locationName,locationCity,personCaseRel,personLocationRel
0,KI,Kenji Iwamura,1959-??-??,Mountaineer,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,Open/Investigated,SOS_SIGN_HOLE,SOS Sign Discovery Site,Hokkaido,VICTIM_IN,FOUND_AT


In [108]:
query_suspect_individual_at_location("KI", "SOS_SIGN_HOLE")

""


In [111]:
query_victim_individual_at_location("CB", "PRAIA_DA_LUZ")

""


In [112]:
query_suspect_individual_at_location("CB", "PRAIA_DA_LUZ")

,personId,personName,personDOB,personType,caseId,caseName,caseStatus,locationId,locationName,locationCity,personCaseRel,personLocationRel
0,CB,Christian Brueckner,1976-12-07,Suspect,CASE_MM,Disappearance of Madeleine McCann,Open (Missing),PRAIA_DA_LUZ,PRAIA_DA_LUZ,Lagos,SUSPECT_IN,PRESENT_NEAR


# Embedding Prediction Layer

In [144]:
def predict_suspect_links_common_neighbors(limit: int = 25) -> pd.DataFrame:
    q = """
    MATCH (p:Person), (c:Case)
    WHERE NOT (p)-[:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c)

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        gds.alpha.linkprediction.commonNeighbors(p, c, {direction: "BOTH"}) AS score
    ORDER BY score DESC, personId, caseId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit})

def predict_suspect_links_adamic_adar(limit: int = 25) -> pd.DataFrame:
    q = """
    MATCH (p:Person), (c:Case)
    WHERE NOT (p)-[:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c)

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        gds.alpha.linkprediction.adamicAdar(p, c, {direction: "BOTH"}) AS score
    ORDER BY score DESC, personId, caseId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit})

def predict_suspect_links_preferential_attachment(limit: int = 25) -> pd.DataFrame:
    q = """
    MATCH (p:Person), (c:Case)
    WHERE NOT (p)-[:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST]->(c)

    RETURN
        p.id AS personId,
        p.name AS personName,
        c.id AS caseId,
        c.name AS caseName,
        gds.alpha.linkprediction.preferentialAttachment(p, c) AS score
    ORDER BY score DESC, personId, caseId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit})

In [145]:
predict_suspect_links_common_neighbors()

,personId,personName,caseId,caseName,score
0,KS,Kristin Smart,CASE_KS,Murder of Kristin Smart,4.0
1,MB,Molly Bish,CASE_MB,Killing of Molly Bish,4.0
2,GM,Gerry McCann,CASE_MM,Disappearance of Madeleine McCann,3.0
3,KM,Kate McCann,CASE_MM,Disappearance of Madeleine McCann,3.0
4,MM,Madeleine McCann,CASE_MM,Disappearance of Madeleine McCann,3.0
5,CA,Cheryl Anderson,CASE_KS,Murder of Kristin Smart,2.0
6,DS,Denise Smart,CASE_KS,Murder of Kristin Smart,2.0
7,JOHN,John J. Bish Sr.,CASE_MB,Killing of Molly Bish,2.0
8,LE,Worcester County DA / State Police,CASE_MB,Killing of Molly Bish,2.0
9,MAGI,"Magdalene ""Magi"" Bish",CASE_MB,Killing of Molly Bish,2.0


In [146]:
predict_suspect_links_adamic_adar()

,personId,personName,caseId,caseName,score
0,MB,Molly Bish,CASE_MB,Killing of Molly Bish,3.984521
1,KS,Kristin Smart,CASE_KS,Murder of Kristin Smart,3.211616
2,GM,Gerry McCann,CASE_MM,Disappearance of Madeleine McCann,2.000806
3,KM,Kate McCann,CASE_MM,Disappearance of Madeleine McCann,2.000806
4,MM,Madeleine McCann,CASE_MM,Disappearance of Madeleine McCann,2.000806
5,LE,Worcester County DA / State Police,CASE_MB,Killing of Molly Bish,1.820478
6,MAGI,"Magdalene ""Magi"" Bish",CASE_MB,Killing of Molly Bish,1.468350
7,KI,Kenji Iwamura,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,1.442695
8,TJGE,Terry Joseph Good Voice Elk,CASE_TJGE,Disappearance of Terry Joseph Good Voice Elk,1.442695
9,DS,Denise Smart,CASE_KS,Murder of Kristin Smart,1.391138


In [147]:
predict_suspect_links_preferential_attachment()

,personId,personName,caseId,caseName,score
0,KS,Kristin Smart,CASE_MB,Killing of Molly Bish,64.0
1,PF,Paul Flores,CASE_MB,Killing of Molly Bish,64.0
2,KS,Kristin Smart,CASE_KS,Murder of Kristin Smart,48.0
3,KS,Kristin Smart,CASE_MM,Disappearance of Madeleine McCann,48.0
4,KS,Kristin Smart,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,48.0
5,MB,Molly Bish,CASE_MB,Killing of Molly Bish,48.0
6,PF,Paul Flores,CASE_MM,Disappearance of Madeleine McCann,48.0
7,PF,Paul Flores,CASE_MOUNTAIN,Mountaineering Incident in Daisetsuzan Nationa...,48.0
8,MB,Molly Bish,CASE_KS,Murder of Kristin Smart,36.0
9,MB,Molly Bish,CASE_MM,Disappearance of Madeleine McCann,36.0


In [131]:
def inspect_person_case_shared_neighbors(limit: int = 20) -> pd.DataFrame:
    q = """
    MATCH (p:Person)-[]-(x)-[]-(c:Case)
    RETURN
        p.id AS personId,
        c.id AS caseId,
        labels(x) AS sharedNeighborLabels,
        x.id AS sharedNeighborId
    LIMIT $limit
    """
    return run_cypher(q, {"limit": limit})

inspect_person_case_shared_neighbors()

,personId,caseId,sharedNeighborLabels,sharedNeighborId
0,TOKYO_MAN_2,CASE_MOUNTAIN,[Location],DAISETSUZAN_PATH
1,TOKYO_MAN_1,CASE_MOUNTAIN,[Location],DAISETSUZAN_PATH
2,KI,CASE_MOUNTAIN,[Location],SOS_SIGN_HOLE
3,CB,CASE_MM,[Location],PRAIA_DA_LUZ
4,KM,CASE_MM,[Location],APT_5A
5,GM,CASE_MM,[Location],APT_5A
6,MM,CASE_MM,[Location],APT_5A
7,MM,CASE_MM,[Person],KM
8,GM,CASE_MM,[Person],KM
9,KM,CASE_MM,[Person],GM


In [141]:
def materialize_case_locations() -> None:
    q = """
    MATCH (p:Person)-[:LAST_SEEN_AT|LAST_SEEN_NEAR|PRESENT_NEAR|FOUND_AT|RESCUED_AT|STAYED_AT]->(l:Location)
    MATCH (p)-[:SUSPECT_IN|ACCUSED_IN|PERSON_OF_INTEREST|VICTIM_IN|MISSING_PERSON_IN|POTENTIAL_VICTIM_IN|RESCUED_IN]->(c:Case)
    MERGE (c)-[:HAS_LOCATION]->(l)
    """
    run_cypher(q)

materialize_case_locations()